# Qwen3-8B 선택적 교체 + 중복 방지 실험 노트북

이번 노트북은 **`short_clean_v3`를 기본 앵커로 고정하고, 정말 필요한 샘플만 선택적으로 교체**하는 1순위 실험입니다.

추가로 중요한 점:
- 제출 파일은 이름만 다르고 내용이 같은 경우가 생길 수 있어서,
- 이번 노트북은 **생성 후 기존 상위 제출본들과 exact same 여부 / 다른 행 개수 / 해시값**을 자동 비교합니다.

생성 파일:
- `submit_30_selective_swap_consensus.csv`
- `submit_31_selective_swap_margin.csv`
- `submit_32_selective_swap_complex_only.csv`
- `submit_30_32_duplicate_compare_report.csv`
- `submit_30_32_preview_report.csv`

추천 순서:
1. `submit_31_selective_swap_margin.csv`
2. `submit_30_selective_swap_consensus.csv`
3. `submit_32_selective_swap_complex_only.csv`


In [ ]:
import sys, pandas as pd
print('python', sys.version)
print('pandas', pd.__version__)


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import hashlib
import re
import pandas as pd
from rouge import Rouge

# ============================================================
# CONFIG
# ============================================================
MANUAL_DIR = Path("/root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/manual_submissions")

CANDIDATES = {
    "short_clean_v3": MANUAL_DIR / "submit_12_top1_absShort_clean_v3.csv",
    "mbr_narrow": MANUAL_DIR / "submit_24_narrow_mbr_best3_clean.csv",
    "abstract_short": MANUAL_DIR / "submit_05_top1_abstract_short.csv",
    "ultra_anchor": MANUAL_DIR / "submit_26_ultra_narrow_mbr_anchor.csv",
}

OUT_FILES = {
    "sel_consensus": MANUAL_DIR / "submit_30_selective_swap_consensus.csv",
    "sel_margin": MANUAL_DIR / "submit_31_selective_swap_margin.csv",
    "sel_complex": MANUAL_DIR / "submit_32_selective_swap_complex_only.csv",
}

ROUGE = Rouge()

# ============================================================
# 유틸
# ============================================================
def normalize_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("/no_think", " ")
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "빈 요약"


def load_submission(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"missing candidate csv: {path}")
    df = pd.read_csv(path)
    if "fname" not in df.columns or "summary" not in df.columns:
        raise ValueError(f"invalid csv format: {path}")
    df = df[["fname", "summary"]].copy()
    df["summary"] = df["summary"].map(normalize_text)
    return df


def score_pair(a: str, b: str) -> float:
    try:
        s = ROUGE.get_scores([normalize_text(a)], [normalize_text(b)])[0]
        return float(s["rouge-1"]["f"])
    except Exception:
        return 0.0


def features_from_text(summary: str) -> dict:
    txt = str(summary)
    return {
        "length": len(txt),
        "comma": txt.count(","),
        "sent_end": txt.count(".") + txt.count("!") + txt.count("?"),
    }


def choose_consensus(sc: str, mn: str, ab: str, ua: str):
    # 기본 anchor = short_clean_v3
    if sc == mn or sc == ua:
        return sc, "keep_anchor_exact_agree"
    # mbr_narrow 와 ultra_anchor 가 같고 둘 다 abstract_short 보다 short_clean_v3와 더 가깝다면 교체
    if mn == ua and mn != sc:
        return mn, "swap_mn_ua_exact_agree"

    agree_sc = (score_pair(sc, mn) + score_pair(sc, ab) + score_pair(sc, ua)) / 3.0
    agree_mn = (score_pair(mn, sc) + score_pair(mn, ab) + score_pair(mn, ua)) / 3.0
    agree_ua = (score_pair(ua, sc) + score_pair(ua, mn) + score_pair(ua, ab)) / 3.0

    if agree_mn >= agree_sc + 0.010 and agree_ua >= agree_sc:
        return mn, "swap_mn_consensus"
    return sc, "keep_anchor_default"


def choose_margin(sc: str, mn: str, ab: str, ua: str):
    agree_sc = (score_pair(sc, mn) + score_pair(sc, ab) + score_pair(sc, ua)) / 3.0
    agree_mn = (score_pair(mn, sc) + score_pair(mn, ab) + score_pair(mn, ua)) / 3.0
    agree_ua = (score_pair(ua, sc) + score_pair(ua, mn) + score_pair(ua, ab)) / 3.0

    # ultra_anchor가 mbr_narrow와 동일하거나 매우 유사하면 mbr_narrow 쪽 가중
    if mn == ua and mn != sc:
        return mn, "swap_margin_exact"

    # agreement margin이 확실한 경우에만 교체
    if agree_mn >= agree_sc + 0.015:
        return mn, "swap_margin_mn"
    if agree_ua >= agree_sc + 0.018:
        return ua, "swap_margin_ua"
    return sc, "keep_margin"


def choose_complex_only(sc: str, mn: str, ab: str, ua: str):
    feat_sc = features_from_text(sc)
    feat_mn = features_from_text(mn)
    agree_sc_mn = score_pair(sc, mn)
    agree_sc_ua = score_pair(sc, ua)

    # anchor가 너무 길고, mbr 쪽이 더 짧고 깔끔하며 anchor와 충분히 다를 때만 교체
    if feat_sc["length"] >= 80 and feat_mn["length"] <= feat_sc["length"] - 8 and agree_sc_mn <= 0.92:
        return mn, "swap_complex_mn"
    if feat_sc["comma"] >= 2 and agree_sc_ua <= 0.90 and len(ua) < len(sc):
        return ua, "swap_complex_ua"
    return sc, "keep_complex"


def save_submission(base_df: pd.DataFrame, summaries, out_path: Path):
    out = pd.DataFrame({"fname": base_df["fname"], "summary": summaries})
    out.to_csv(out_path, index=False)
    print("saved:", out_path)


def df_hash(df: pd.DataFrame) -> str:
    text = "\n".join((df["fname"].astype(str) + "\t" + df["summary"].astype(str)).tolist())
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def compare_against_existing(name: str, new_df: pd.DataFrame, loaded: dict):
    rows = []
    new_hash = df_hash(new_df)
    for prev_name, prev_df in loaded.items():
        same = new_df.equals(prev_df)
        diff_cnt = int((new_df["summary"].fillna("") != prev_df["summary"].fillna("")).sum())
        rows.append({
            "new_submission": name,
            "compare_to": prev_name,
            "exact_same": same,
            "different_rows": diff_cnt,
            "new_hash": new_hash,
            "prev_hash": df_hash(prev_df),
        })
    return pd.DataFrame(rows)


# ============================================================
# 로딩
# ============================================================
loaded = {name: load_submission(path) for name, path in CANDIDATES.items()}
base = loaded["short_clean_v3"][["fname"]].copy()
for name, df in loaded.items():
    if len(df) != len(base):
        raise ValueError(f"length mismatch: {name}")
    if df["fname"].tolist() != base["fname"].tolist():
        raise ValueError(f"fname order mismatch: {name}")

# ============================================================
# 선택적 교체 실험
# ============================================================
plans = {
    "sel_consensus": choose_consensus,
    "sel_margin": choose_margin,
    "sel_complex": choose_complex_only,
}

all_compare = []
all_preview = []
for plan_name, chooser in plans.items():
    picked = []
    reasons = []
    counts = {"short_clean_v3": 0, "mbr_narrow": 0, "ultra_anchor": 0}

    for i in range(len(base)):
        sc = loaded["short_clean_v3"].iloc[i]["summary"]
        mn = loaded["mbr_narrow"].iloc[i]["summary"]
        ab = loaded["abstract_short"].iloc[i]["summary"]
        ua = loaded["ultra_anchor"].iloc[i]["summary"]

        chosen, reason = chooser(sc, mn, ab, ua)
        if chosen == mn:
            counts["mbr_narrow"] += 1
        elif chosen == ua:
            counts["ultra_anchor"] += 1
        else:
            counts["short_clean_v3"] += 1
        picked.append(chosen)
        reasons.append(reason)

    out_df = pd.DataFrame({"fname": base["fname"], "summary": picked})
    save_submission(base, picked, OUT_FILES[plan_name])
    print(plan_name, "selected counts:", counts)

    comp = compare_against_existing(plan_name, out_df, loaded)
    all_compare.append(comp)

    preview = pd.DataFrame({
        "submission": plan_name,
        "fname": base["fname"].head(12),
        "reason": reasons[:12],
        "summary": picked[:12],
    })
    all_preview.append(preview)

compare_df = pd.concat(all_compare, ignore_index=True)
preview_df = pd.concat(all_preview, ignore_index=True)

compare_path = MANUAL_DIR / "submit_30_32_duplicate_compare_report.csv"
preview_path = MANUAL_DIR / "submit_30_32_preview_report.csv"
compare_df.to_csv(compare_path, index=False)
preview_df.to_csv(preview_path, index=False)

print("\n===== duplicate compare report =====")
print(compare_df)
print("saved:", compare_path)

print("\n===== preview report =====")
print(preview_df)
print("saved:", preview_path)
